# 01 · Exploratory Data Analysis (M4)
Goal: understand the data and let the evidence justify our modelling choices. Run `python run_pipeline.py --only clean` first so the cleaned dataset exists.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))          # lets the notebook import the project's src/ package
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)

In [ ]:
df = pd.read_parquet(ROOT/"data/processed/flowcast_clean.parquet")
print(df.shape)
df.head()

## 1. Descriptive statistics
Mean, median, standard deviation, variance, skewness. A skew > 0 means a long right tail (a few very busy windows).

In [ ]:
num = ["traffic_volume","avg_speed","occupancy","travel_time"]
desc = df[num].agg(["mean","median","std","var","skew","kurt"]).T
desc["mode"] = [df[c].round().mode().iat[0] for c in num]
desc.round(2)

## 2. Run the full EDA module (saves all figures to `reports/figures/`)

In [ ]:
from src import eda
_ = eda.run()
for f in ["eda_distributions","eda_correlation","eda_hourly_pattern","eda_congestion_heatmap","eda_weather_accident","eda_class_balance"]:
    display(Image(str(ROOT/"reports/figures"/f"{f}.png")))

## 3. What the evidence tells us
* **Two daily peaks** (morning and evening) and a different weekend shape → we engineer hour/day-of-week encodings, a peak flag and a weekend flag.
* **Volume and occupancy are strongly correlated; speed and occupancy are strongly negatively correlated** → redundancy; tree models cope well, linear models need scaling.
* **Congestion classes are imbalanced** (Free-flow dominates) → accuracy would mislead; we use **macro-F1**.
* **Rain/Fog multiply the incident rate** and heavy congestion raises it too → weather flags and V/C ratio are features for accident risk.

In [ ]:
Markdown((ROOT/"reports/eda_summary.md").read_text())